# Hreal: double slit without PML

This notebook preserves the normalized plane-wave Hamiltonian of the original
`Hreal` calculation.  Its x and y boundaries are periodic because the basis is
global.  The helper module keeps the original equations independently so the
clean implementation can be regression-tested against them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from double_slit_pml.model import (
    PlaneWaveModel, build_hamiltonian, compact_packet,
    evolve, project_separable_state, reconstruct,
)
from double_slit_pml.legacy_hreal import legacy_hreal_matrix

In [ ]:
# Original Hreal parameters
model = PlaneWaveModel(Lx=1.0, Ly=1.0, nx=6, ny=6)
H = build_hamiltonian(model)

# Exact algebraic regression against the preserved notebook expression
np.max(np.abs(H.dense() - legacy_hreal_matrix(nx=6, ny=6)))

In [ ]:
eigenvalues = np.linalg.eigvalsh(H.dense())[:10]
eigenvalues

In [ ]:
psi0 = project_separable_state(model, lambda x: compact_packet(x, k0=12.0))
times = np.linspace(0.0, 0.15, 4)
states = evolve(model, psi0, times, H)

x = np.linspace(-1.0, 1.0, 240)
y = np.linspace(-1.0, 1.0, 180)
fig, axes = plt.subplots(1, len(times), figsize=(12, 3), sharex=True, sharey=True)
for ax, state, time in zip(axes, states, times):
    density = np.abs(reconstruct(model, state, x, y))**2
    ax.imshow(density.T, origin='lower', extent=(-1, 1, -1, 1), aspect='auto', cmap='magma')
    ax.set_title(f't = {time:.2f}')
    ax.set_xlabel('x')
axes[0].set_ylabel('y')
fig.suptitle('Hreal: periodic plane-wave box, no PML')
plt.tight_layout()